In [10]:
# region Imports

#* --------------------------------------------------------------------------------
#* General purpose imports
#* --------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import fisher_exact, barnard_exact, ranksums
from matplotlib.ticker import FuncFormatter
import pickle as pkl


#* --------------------------------------------------------------------------------
#* Personal librairies imports
#* --------------------------------------------------------------------------------
import sys, os
src_path = os.path.abspath(os.path.join("..", "src"))
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from utils import astro_utils as au
from utils import maths_utils  as mu
from utils import stats_utils  as su
from utils import spherical_utils  as spu
from utils import graphics_utils  as gu
from utils import labels_utils  as lu
from utils import pandas_utils  as pu
from utils import physics_utils  as phu


#* --------------------------------------------------------------------------------
#* Project modules imports
#* --------------------------------------------------------------------------------
import sSFR
import generate_report as report
import domination

#* --------------------------------------------------------------------------------
#* Global variables
#* --------------------------------------------------------------------------------
import config as co

#* --------------------------------------------------------------------------------
#* Project data
#* --------------------------------------------------------------------------------

with open(co.DATA_PATH + co.PROCESS_SAMPLES, "rb") as file:
            sample = pkl.load(file)



# endregion

In [2]:
CG_Bozzio_Gals = pd.read_csv(co.DATA_PATH + "CG_Bozzio.csv")

In [3]:
CG_Bozzio_Gals.rename(columns={'zsp':'z', 'CG':'Group', '_RAJ2000':'RA', '_DEJ2000':'Dec'}, inplace=True)

In [31]:
CG_Bozzio_Gals

,#angDist,RA,Dec,Group,Gal,Morph,PS,PE,PS0,Kmag,...,e_pmRA,pmDE,e_pmDE,SpObjID,z,e_zsp,f_zsp,spType,spCl,subClass
0,0.037525,0.19571,28.40202,1,1,E,0.0,0.8,0.2,10.41,...,2.0,-11.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,0.223224,0.15808,28.38454,1,2,S,1.0,0.0,0.0,10.46,...,3.0,20.0,3.0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,0.113664,0.18333,28.40145,1,3,S,1.0,0.0,0.0,11.53,...,3.0,-4.0,3.0,0,NaN,NaN,NaN,NaN,NaN,NaN
3,0.166910,0.17671,28.36901,1,4,E,0.0,0.8,0.2,13.26,...,2.0,3.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,0.326517,0.18787,28.37172,1,5,S,1.0,0.0,0.0,13.40,...,2.0,-1.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,0.165263,358.47450,7.97055,85,1,S,1.0,0.0,0.0,9.15,...,4.0,-32.0,4.0,0,NaN,NaN,NaN,NaN,NaN,NaN
191,0.194496,358.36162,7.87566,85,2,S,1.0,0.0,0.0,9.52,...,2.0,-3.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN
192,0.058653,358.33192,7.87093,85,3,E,0.0,0.8,0.2,9.71,...,2.0,2.0,2.0,0,NaN,NaN,NaN,NaN,NaN,NaN
193,0.181612,358.38396,8.11813,85,4,S,1.0,0.0,0.0,10.66,...,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
def Velocity(z,zg):
    return c*(z-zg)/(1+zg)

from astropy.cosmology import FlatLambdaCDM
from astropy import units as u

def Group_agg(x):
          
    # Planck 2015 Cosmological Parameters
    H0 = 67.8  # Hubble constant in km/s/Mpc
    h = H0/100
    Om0 = 0.308  # Matter density parameter
    Tcmb0 = 2.7255  # CMB temperature today in Kelvin (default for Planck)
    Neff = 3.15  # Effective number of neutrino species
    
    # Initialize the FlatLambdaCDM cosmology
    cosmo = FlatLambdaCDM(H0=H0, Om0=Om0, Tcmb0=Tcmb0, Neff=Neff)

    Nb_Gal = len(x)
    z_group = x['z'].mean()
    Radius_Bary_arcmin = spu.calc_diameter_arcmin(x)   
    arcmin_to_rad = np.pi/(180*60)
    Dist_Group_Mpc = cosmo.luminosity_distance(z_group)
    size_Group_Bary_kpc =  (Radius_Bary_arcmin * arcmin_to_rad * Dist_Group_Mpc).to(u.kpc).value
    Vdisp = su.V_disp_gapper(x)
    t_cr = phu.crossing_time(size_Group_Bary_kpc,Vdisp)
    frac_S = len(x[x['Morph'] == 'S'])/Nb_Gal

    values = [Radius_Bary_arcmin, Vdisp, t_cr, Nb_Gal, frac_S] 
    labels = ['Radius_Bary_arcmin', 'Vdisp', 't_cr', 'Nb_Gal', 'Frac_S']
    
        
    return pd.Series(values, index=labels) 


In [33]:
CG_Bozzio_Groups = CG_Bozzio_Gals.groupby('Group').apply(Group_agg).reset_index()

/var/folders/2l/5qdg19zx40sdthfzpld2xpq80000gn/T/ipykernel_13527/794469716.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CG_Bozzio_Groups = CG_Bozzio_Gals.groupby('Group').apply(Group_agg).reset_index()


In [34]:
CG_Bozzio_Groups

,Group,Radius_Bary_arcmin,Vdisp,t_cr,Nb_Gal,Frac_S
0,1,1.775236,NaN,NaN,5.0,0.600000
1,3,4.128930,NaN,NaN,5.0,0.400000
2,4,3.786373,137.552596,0.445739,4.0,0.750000
3,6,5.639184,NaN,NaN,6.0,1.000000
4,9,6.810920,NaN,NaN,4.0,0.750000
5,11,4.475187,NaN,NaN,4.0,0.250000
6,13,9.403869,NaN,NaN,4.0,0.750000
7,20,2.687861,86.303400,0.759321,4.0,0.750000
8,21,4.044236,151.974974,1.070316,4.0,1.000000
9,23,5.636524,NaN,NaN,4.0,0.500000


In [29]:
from scipy.stats import spearmanr

In [39]:

df = CG_Bozzio_Groups
for q in ['Radius_Bary_arcmin', 'Vdisp', 't_cr']:
    print(f"{q}")
    df_clean = df[[q, 'Frac_S']].dropna()
    spearcorr, spearpval = spearmanr(df_clean[q], df_clean['Frac_S'])
    # if spearpval < 0.05:
    print(f"   p-value={spearpval:.3f}      statistic={spearcorr:.2f}")


Radius_Bary_arcmin
   p-value=0.423      statistic=0.12
Vdisp
   p-value=0.546      statistic=-0.16
t_cr
   p-value=0.914      statistic=0.03
